# Baseline Modelling — Polymarket → Gold Returns

Pipeline summary (this notebook covers steps 1-9 of the design brief):

1. **Configure** folders, data sources, feature-engineering toggles, and model hyperparameters.
2. **Load** the Bloomberg target (`GOLD USD SPOT PER OZ`, `Close`) and resolve the latest Polymarket input panel.
3. **Build** the regression target `y` as future gold log-returns.
4. **Engineer and align** AR features plus the Bloomberg/Polymarket design matrix on the 5-minute grid.
5. **Construct** the traditional, combined, and ablation dataset variants in `DATASET_SPECS`.
6. **Define** the modelling framework: shared model factories plus the walk-forward runner.
7. **Walk-forward evaluation** (train on window → predict next observation, or next 12 for LSTM-family models (`lstm` and `lstm_pls`)).
8. **Logging**: every run appends a row to `Results/runs_log.txt` plus a detailed `.json` side-car (model spec, metrics, data hash, run timestamp).
9. **Persistence and summary**: models are saved as `Models/<MODEL>_<WINDOW>_<DATASET>_h<HORIZON>m_f<NFEAT>_<DATA-DATE>.pkl|.keras`, cached runs are reused when inputs match, and a final summary table is written to `Results/`.

> **Open questions / critical issues are collected in `NOTES_modelling_baseline.md` — please read it before interpreting results.**

## 0. Config

In [1]:
# %% ── CELL 0 : CONFIG ───────────────────────────────────────────────────────
from pathlib import Path
import json
import re

# Folders
DATA_DIR     = Path("./Data")
MODELS_DIR   = Path("./Models")
RESULTS_DIR  = Path("./Results")
for p in (MODELS_DIR, RESULTS_DIR):
    p.mkdir(exist_ok=True, parents=True)

# File patterns
GOLD_PANEL_GLOB = "gold_panel_*.csv"
BLOOMBERG_XLSX_CANDIDATES = [
    DATA_DIR / "Indicators Data bloomberg.xlsx",
    DATA_DIR / "Indicators-Data-bloomberg.xlsx",
    DATA_DIR / "Indicators_Data_bloomberg.xlsx",
]
TARGET_SHEET = "GOLD USD SPOT PER OZ"
TARGET_COL   = "Close"

TRADITIONAL_STATIONARITY_MODE = "test_driven"  # "test_driven" reads bloomberg_panel_stationary_*.csv; "blanket" preserves the legacy make_stationary_features path
BB_STATIONARY_PANEL_GLOB      = "bloomberg_panel_stationary_*.csv"
BB_STATIONARITY_METADATA_GLOB = "bloomberg_stationarity_metadata_*.json"

BB_STATIONARY_PANEL_PATH      = None
BB_STATIONARITY_METADATA_PATH = None
BB_STATIONARITY_METADATA      = None
BB_STATIONARITY_DATE          = None

def resolve_bloomberg_xlsx(candidates: list[Path], data_dir: Path) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate
    wildcard = sorted(data_dir.glob("*Indicators*Data*bloomberg*.xlsx"))
    if wildcard:
        return wildcard[0]
    tried = "\n  - ".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(
        "Bloomberg workbook not found. Tried:\n"
        f"  - {tried}\n"
        "Expected something like 'Indicators Data bloomberg.xlsx' in ./Data."
    )

def resolve_latest_bloomberg_stationarity_artefacts(data_dir: Path) -> tuple[Path, Path, str]:
    panel_rx = re.compile(r"bloomberg_panel_stationary_(\d{4}-\d{2}-\d{2})\.csv$")
    metadata_rx = re.compile(r"bloomberg_stationarity_metadata_(\d{4}-\d{2}-\d{2})\.json$")

    panel_candidates = []
    for path in data_dir.glob(BB_STATIONARY_PANEL_GLOB):
        match = panel_rx.search(path.name)
        if match:
            panel_candidates.append((match.group(1), path))

    if not panel_candidates:
        raise FileNotFoundError(
            f"No files matching {BB_STATIONARY_PANEL_GLOB} in {data_dir}.\n"
            "Run the PLS Feature Engineering notebook first to generate the Bloomberg stationary panel."
        )

    metadata_by_date = {}
    for path in data_dir.glob(BB_STATIONARITY_METADATA_GLOB):
        match = metadata_rx.search(path.name)
        if match:
            metadata_by_date[match.group(1)] = path

    panel_candidates.sort(key=lambda item: item[0])
    latest_date, panel_path = panel_candidates[-1]
    metadata_path = metadata_by_date.get(latest_date)
    if metadata_path is None:
        raise FileNotFoundError(
            f"Missing matching bloomberg_stationarity_metadata_{{date}}.json for {panel_path.name}."
        )

    return panel_path, metadata_path, latest_date

BLOOMBERG_XLSX = resolve_bloomberg_xlsx(BLOOMBERG_XLSX_CANDIDATES, DATA_DIR)
print(f"Using Bloomberg workbook: {BLOOMBERG_XLSX}")

# ─── Modelling ───────────────────────────────────────────────────────────────
BAR_MINUTES        = 5
RETURN_HORIZON_MIN = 60
HORIZON_STEPS      = max(1, RETURN_HORIZON_MIN // BAR_MINUTES)
AR_LAGS            = [1, 2, 3, 6, 12]
AR_MA_WINDOWS      = [3, 6, 12, 36]
WINDOW_SCHEMES = {
    #"fixed120":  ("fixed", 120),
    "fixed240":  ("fixed", 240),
    #"fixed300":  ("fixed", 300),
    "expanding": ("expanding", 120)
}
LSTM_TEST_BLOCK = 12
LSTM_SEQ_LEN    = 12
LSTM_EPOCHS     = 8
LSTM_BATCH      = 32
RF_N_ESTIMATORS = 200
RANDOM_STATE    = 67

# ─── Traditional indicators ─────────────────────────────────────────────────
TRADITIONAL_MAX_FFILL_BARS    = None
TRADITIONAL_USE_STALENESS     = True
TRADITIONAL_DROP_PRICE_LEVELS = True
TRADITIONAL_AR_LAGS           = [1, 2, 3, 6, 12]

# ─── Feature engineering gate ───────────────────────────────────────────────
FEATURE_ENGINEERING_DONE = True  # Set to True to skip feature engineering and load pre-engineered panels
MAX_FEATURES_PREFILTER   = 150

# ─── Daily gap handling ─────────────────────────────────────────────────────
HANDLE_DAILY_GAP = True
GAP_THRESHOLD    = "60min"

# ─── Feature-engineering input mode ─────────────────────────────────────────
# "raw_panel" keeps the filtered polymarket panel in its native feature space and
# enables in-window PLS / Lasso preprocessing inside model wrappers.
# "preprocessed" expects an already-engineered low-dimensional panel and therefore
# disables the PLS variants to avoid double-reduction.
FE_INPUT_MODE = "raw_panel"

# Sensitivity-check controls for alternative PLS component counts.
RUN_PLS_SENSITIVITY_SWEEP = False
PLS_SENSITIVITY_GRID      = [3, 5, 8, 10]

# ── Dataset toggles ──────────────────────────────────────────────────────────
RUN_POLY           = True
RUN_TRAD           = True
RUN_COMBINED       = True
RUN_POLY_ONLY      = True
RUN_BLOOMBERG_ONLY = True
RUN_AR_ONLY        = True

# ── Model toggles ────────────────────────────────────────────────────────────
RUN_LINEAR    = False   # OLS, no dim-reduction (used on low-dim datasets)
RUN_RF        = False   # raw-feature Random Forest
RUN_LSTM      = False   # raw-feature LSTM
RUN_PLS_OLS   = False   # PLS-on-poly-block + passthrough + OLS readout
RUN_RF_PLS    = False   # PLS-on-poly-block + passthrough + Random Forest
RUN_LSTM_PLS  = True   # PLS-on-poly-block + passthrough + LSTM
RUN_LASSO_CV  = False   # LassoCV with internal scaling (dataset-agnostic)

# Evaluation toggles
FORCE_RETRAIN = False
DRY_RUN_ROWS  = None

# %% ── CELL 0.1 : LOAD PLS n_components FROM FE ARTEFACT ─────────────────────
PLS_SELECTION_LOG_DIR = Path("Feature selection log")
PLS_SELECTION_ARTEFACT_PATH = PLS_SELECTION_LOG_DIR / "pls_n_components_latest.json"

if not PLS_SELECTION_ARTEFACT_PATH.exists():
    raise FileNotFoundError(
        "Missing PLS n_components artefact at "
        f"{PLS_SELECTION_ARTEFACT_PATH.resolve()}. "
        "Run 'PLS - Feature Engineering SQL polymarket database.ipynb' first "
        "to generate Feature selection log/pls_n_components_latest.json."
    )

PLS_SELECTION_METADATA = json.loads(
    PLS_SELECTION_ARTEFACT_PATH.read_text(encoding="utf-8")
)
_loaded_pls_n_components = PLS_SELECTION_METADATA.get("n_components")
if (
    isinstance(_loaded_pls_n_components, bool)
    or not isinstance(_loaded_pls_n_components, int)
    or _loaded_pls_n_components < 1
):
    raise ValueError(
        "Invalid PLS selection artefact: expected an integer n_components >= 1, "
        f"got {_loaded_pls_n_components!r}."
    )

PLS_N_COMPONENTS = _loaded_pls_n_components
print(f"Loaded PLS_N_COMPONENTS   : {PLS_N_COMPONENTS}")
print(f"Artefact path             : {PLS_SELECTION_ARTEFACT_PATH}")
print(
    "Artefact generated_at_utc : ",
    PLS_SELECTION_METADATA.get("generated_at_utc", "<missing>")
)
print(
    "Artefact grid_searched    : ",
    PLS_SELECTION_METADATA.get("grid_searched", "<missing>")
)

Using Bloomberg workbook: Data\Indicators Data bloomberg.xlsx
Loaded PLS_N_COMPONENTS   : 3
Artefact path             : Feature selection log\pls_n_components_latest.json
Artefact generated_at_utc :  2026-05-19T12:53:00Z
Artefact grid_searched    :  [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


## 1. Dependencies

In [2]:
# %% ── CELL 1 : DEPENDENCIES ─────────────────────────────────────────────────
import os, re, json, hashlib, joblib, warnings, datetime as dt, time
import numpy as np
import pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.feature_selection import VarianceThreshold

# TensorFlow / Keras (LSTM) — imported lazily so the notebook still runs if TF isn't installed
_TF_AVAILABLE = True
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential, load_model
    from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
    from tensorflow.keras.callbacks import EarlyStopping
    tf.random.set_seed(RANDOM_STATE)
except Exception as _tf_err:
    _TF_AVAILABLE = False
    print("⚠️  TensorFlow not importable — LSTM runs will be skipped.", _tf_err)

np.random.seed(RANDOM_STATE)
print("✅ Dependencies loaded. TF:", _TF_AVAILABLE)


✅ Dependencies loaded. TF: True


## 1.1 Traditional predictor helpers

In [3]:
# %% ── CELL 1.1 : TRADITIONAL PREDICTOR HELPERS ─────────────────────────────

def make_stationary_features(df: pd.DataFrame, exclude_cols=None) -> pd.DataFrame:
    """Convert each column to a stationary series:
    - strictly positive series  → log-diff  (captures percentage moves)
    - other numeric series      → simple diff (rates, spreads, indices that cross 0)
    Columns in `exclude_cols` (e.g. staleness counters) are passed through as-is.
    """
    exclude_cols = set(exclude_cols or [])
    out = pd.DataFrame(index=df.index)
    for col in df.columns:
        s = pd.to_numeric(df[col], errors="coerce")
        if col in exclude_cols:
            out[col] = s
            continue
        strictly_positive = (s.dropna() > 0).all()
        if strictly_positive:
            out[f"{col}_logret"] = np.log(s).diff()
        else:
            out[f"{col}_diff"] = s.diff()
    return out


def build_X_traditional(
    bb_dict: dict,
    target_sheet: str,
    master_index: pd.DatetimeIndex,
    align_index: pd.DatetimeIndex,
    max_ffill_bars=None,
    use_staleness: bool = True,
    ar_lags=(1, 2, 3, 6, 12),
    ar_ma_windows=(3, 6, 12, 36),
    ar_vol_windows=(12, 36),
    logret_bar: pd.Series = None,
    stationarity_mode=None,
    stationary_panel_path=None,
    stationary_metadata=None,
) -> pd.DataFrame:
    """Build a traditional predictor matrix aligned to `align_index`.

    `stationarity_mode="blanket"` preserves the legacy per-sheet workflow:
    1. Reindex the close column to `master_index` (5-min gold clock).
    2. Optionally compute a staleness counter (bars since last real tick).
    3. Forward-fill up to `max_ffill_bars`.
    4. Stationarise: log-diff (positive series) or simple diff (other).
    5. Append AR lags, rolling means, and rolling stds of the gold log-return
       (same full AR feature set as build_ar_features for the polymarket block).
    6. Reindex everything to `align_index` (= X.index, valid polymarket rows).

    `stationarity_mode="test_driven"` instead loads the prebuilt Bloomberg
    stationary panel from the feature-engineering notebook, then appends the
    same AR features and final alignment as the blanket path.
    """
    if stationarity_mode is None:
        stationarity_mode = TRADITIONAL_STATIONARITY_MODE
    if stationary_panel_path is None:
        stationary_panel_path = BB_STATIONARY_PANEL_PATH
    if stationary_metadata is None:
        stationary_metadata = BB_STATIONARITY_METADATA

    if stationarity_mode == "test_driven":
        if stationary_panel_path is None:
            raise FileNotFoundError(
                "TRADITIONAL_STATIONARITY_MODE='test_driven' but no Bloomberg stationary panel was loaded."
            )

        stationary = (
            pd.read_csv(stationary_panel_path, parse_dates=["Date"])
            .set_index("Date")
            .sort_index()
        )
        stationary = stationary.reindex(master_index)

        if stationary_metadata is not None:
            _metadata_target_sheet = stationary_metadata.get("target_sheet")
            if _metadata_target_sheet not in (None, target_sheet):
                print(
                    f"⚠️  Bloomberg stationarity metadata target_sheet={_metadata_target_sheet!r} "
                    f"does not match target_sheet={target_sheet!r}."
                )

        if not use_staleness:
            stationary = stationary.loc[:, [
                col for col in stationary.columns
                if not col.endswith("_stale")
            ]]

    elif stationarity_mode == "blanket":
        import re as _re
        price_cols_raw = {}   # {feature_name: forward-filled Series}
        stale_cols_raw = {}   # {stale_col_name: counter Series}

        for sheet, df in bb_dict.items():
            if sheet == target_sheet:
                continue
            if "close" not in df.columns:
                continue

            feature_name = _re.sub(r"[^0-9a-zA-Z]+", "_", sheet).strip("_").lower()
            s_raw = df["close"].sort_index()
            s_raw = s_raw[~s_raw.index.duplicated(keep="last")]

            # Align to 5-min master index
            s_aligned = s_raw.reindex(master_index)

            if use_staleness:
                # Count bars since the last real (non-NaN) tick
                is_new_tick       = s_aligned.notna()
                real_tick_groups  = is_new_tick.cumsum()
                stale_count       = (~is_new_tick).groupby(real_tick_groups).cumsum().astype(int)
                stale_cols_raw[f"{feature_name}_stale"] = stale_count

            s_filled = s_aligned.ffill(limit=max_ffill_bars)
            price_cols_raw[feature_name] = s_filled

        if not price_cols_raw:
            return pd.DataFrame(index=align_index)

        price_df = pd.DataFrame(price_cols_raw, index=master_index)
        stale_df = pd.DataFrame(stale_cols_raw, index=master_index)

        # Stationarise price-level columns
        stationary = make_stationary_features(price_df)

        # Merge staleness counters back (they stay in levels — already stationary-ish)
        if use_staleness and not stale_df.empty:
            stationary = stationary.join(stale_df, how="left")

    else:
        raise ValueError(
            f"Unknown TRADITIONAL_STATIONARITY_MODE: {stationarity_mode!r}"
        )

    # Add the full AR feature set for the gold log-return — mirrors build_ar_features
    # used for the polymarket block (5 lagged returns, 4 rolling means, 2 rolling stds).
    if logret_bar is not None:
        logret_aligned = logret_bar.reindex(master_index)
        for lag in ar_lags:
            stationary[f"trad_gold_lag{lag}"] = logret_aligned.shift(lag)
        for w in ar_ma_windows:
            stationary[f"trad_gold_ma{w}"] = logret_aligned.shift(1).rolling(w).mean()
        for w in ar_vol_windows:
            stationary[f"trad_gold_std{w}"] = logret_aligned.shift(1).rolling(w).std()

    # Final alignment to the modelling index
    stationary = stationary.reindex(align_index)
    stationary = stationary.replace([np.inf, -np.inf], np.nan)
    stationary = stationary.ffill().fillna(0.0)   # 0.0 for series-start NaN

    return stationary


print("✅ Traditional predictor helpers loaded.")

✅ Traditional predictor helpers loaded.


## 2. Load latest gold panel

In [4]:
# %% ── CELL 2 : LOAD LATEST GOLD PANEL ───────────────────────────────────────
_date_rx = re.compile(r"gold_panel_(\d{4}-\d{2}-\d{2})\.csv$")

def find_latest_panel(folder: Path) -> tuple[Path, str]:
    candidates = []
    for p in folder.glob(GOLD_PANEL_GLOB):
        m = _date_rx.search(p.name)
        if m:
            candidates.append((m.group(1), p))
    if not candidates:
        raise FileNotFoundError(f"No files matching {GOLD_PANEL_GLOB} in {folder}")
    candidates.sort(key=lambda t: t[0])   # lexicographic sort works for ISO dates
    return candidates[-1][1], candidates[-1][0]

# ── Select panel file based on FE_INPUT_MODE ──────────────────────────────────
if FE_INPUT_MODE == "raw_panel":
    # Primary output of the PLS Feature Engineering notebook after all filters.
    # AR features are NOT included — this notebook builds them from scratch.
    _poly_rx   = re.compile(r"polymarket_panel_filtered_(\d{4}-\d{2}-\d{2})\.csv$")
    _poly_glob = "polymarket_panel_filtered_*.csv"

    def _find_latest_filtered_panel(folder: Path) -> tuple[Path, str]:
        candidates = []
        for p in folder.glob(_poly_glob):
            m = _poly_rx.search(p.name)
            if m:
                candidates.append((m.group(1), p))
        if not candidates:
            raise FileNotFoundError(
                f"No polymarket_panel_filtered_*.csv found in {folder}.\n"
                "Run the PLS Feature Engineering notebook first, or set "
                "FE_INPUT_MODE='preprocessed' only to use the dormant legacy pre-engineered-panel branch."
            )
        candidates.sort(key=lambda t: t[0])
        return candidates[-1][1], candidates[-1][0]

    panel_path, panel_date = _find_latest_filtered_panel(DATA_DIR)

elif FE_INPUT_MODE == "preprocessed":
    # Placeholder path for the dormant legacy pre-engineered-panel branch.
    # Keep this for backwards compatibility; replace the filename only if reviving that workflow.
    # script generates its output.
    panel_path = DATA_DIR / "polymarket_panel_preprocessed_YYYY-MM-DD.csv"  # TODO: update
    panel_date = "unknown"
    if not panel_path.exists():
        raise FileNotFoundError(
            f"Preprocessed panel not found: {panel_path}\n"
            "Set FE_INPUT_MODE='raw_panel' or place the legacy preprocessed CSV at the above path."
        )

else:
    raise ValueError(f"Unknown FE_INPUT_MODE: {FE_INPUT_MODE!r}")

print(f"FE_INPUT_MODE  : {FE_INPUT_MODE}")
print(f"Latest panel   : {panel_path.name}  (data date = {panel_date})")

X_raw = pd.read_csv(panel_path, parse_dates=["scraped_at"]).set_index("scraped_at").sort_index()
if DRY_RUN_ROWS:
    X_raw = X_raw.iloc[-DRY_RUN_ROWS:]
print(f"X_raw shape    : {X_raw.shape}   range: {X_raw.index.min()} → {X_raw.index.max()}")

if TRADITIONAL_STATIONARITY_MODE == "test_driven":
    BB_STATIONARY_PANEL_PATH, BB_STATIONARITY_METADATA_PATH, BB_STATIONARITY_DATE = (
        resolve_latest_bloomberg_stationarity_artefacts(DATA_DIR)
    )
    print(
        f"Latest Bloomberg stationary panel : {BB_STATIONARY_PANEL_PATH.name}  "
        f"(data date = {BB_STATIONARITY_DATE})"
    )
    print(f"Latest Bloomberg metadata        : {BB_STATIONARITY_METADATA_PATH.name}")

    BB_STATIONARITY_METADATA = json.loads(
        BB_STATIONARITY_METADATA_PATH.read_text(encoding="utf-8")
    )

    if panel_date != "unknown" and BB_STATIONARITY_DATE != panel_date:
        print(
            f"⚠️  Bloomberg stationary artefact date ({BB_STATIONARITY_DATE}) "
            f"does not match polymarket panel date ({panel_date})."
        )

    _bb_transforms = BB_STATIONARITY_METADATA.get("transforms", {})
    _bb_logret_count = sum(transform == "logret" for transform in _bb_transforms.values())
    _bb_diff_count = sum(transform == "diff" for transform in _bb_transforms.values())
    _bb_level_count = sum(transform == "level" for transform in _bb_transforms.values())
    print(
        f"Bloomberg stationarity: {_bb_logret_count + _bb_diff_count} differenced "
        f"({_bb_logret_count} logret, {_bb_diff_count} diff), {_bb_level_count} kept in levels  | "
        f"train_fraction={BB_STATIONARITY_METADATA.get('train_fraction')} | "
        f"max_ffill_bars={BB_STATIONARITY_METADATA.get('max_ffill_bars')}"
    )
elif TRADITIONAL_STATIONARITY_MODE == "blanket":
    BB_STATIONARY_PANEL_PATH = None
    BB_STATIONARITY_METADATA_PATH = None
    BB_STATIONARITY_METADATA = None
    BB_STATIONARITY_DATE = None
else:
    raise ValueError(
        f"Unknown TRADITIONAL_STATIONARITY_MODE: {TRADITIONAL_STATIONARITY_MODE!r}"
    )

FE_INPUT_MODE  : raw_panel
Latest panel   : polymarket_panel_filtered_2026-05-07.csv  (data date = 2026-05-07)
X_raw shape    : (7171, 690)   range: 2026-04-01 00:00:00 → 2026-05-07 22:00:00
Latest Bloomberg stationary panel : bloomberg_panel_stationary_2026-05-07.csv  (data date = 2026-05-07)
Latest Bloomberg metadata        : bloomberg_stationarity_metadata_2026-05-07.json
Bloomberg stationarity: 9 differenced (9 logret, 0 diff), 0 kept in levels  | train_fraction=1.0 | max_ffill_bars=None


In [5]:
# %% ── CELL 3.0 : ENSURE OPENPYXL ───────────────────────────────────────────
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("openpyxl") is None:
    print("openpyxl not found. Installing via pip...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])
else:
    print("openpyxl is already installed.")


openpyxl is already installed.


## 3. Build the target `y`

We want a **regression** target on **future returns**, not prices.

If `H = HORIZON_STEPS`, then the target is the log-return realised over the next `H` bars:

\[
y_t = \log\!\left(\frac{P_{t+H}}{P_t}\right)
\]

implemented as the `H`-bar log-return series shifted by `-H`. Switching `RETURN_HORIZON_MIN` in the config cell updates the target horizon everywhere downstream.

**Why log-returns vs simple returns?** Additivity across time (log-returns sum), symmetry, and far more stable numerics for small moves — standard in high-frequency finance.

### 3.1 Auto-regressive (AR) features

**Design choice — rationale:**
Gold returns at 5-min cadence exhibit small but exploitable **short-term momentum / mean-reversion** and strong **volatility clustering**. Any non-trivial baseline must give the model access to its own past, otherwise we're asking Polymarket features alone to beat a signal that's already in the price path.

I'm adding three groups of AR features, all computed on `logret_bar` (which is *already observed* at time `t`, so no look-ahead):

1. **Lagged returns** at 1, 2, 3, 6, 12 bars (5, 10, 15, 30, 60 min). Captures raw momentum.
2. **Rolling means of past returns** over 3, 6, 12, 36 bars. Same idea as the lectures' moving-average baseline; a denoised direction estimate.
3. **Rolling std (realised vol proxy)** over 12 and 36 bars. Lets tree models condition on the current vol regime — classic HAR-RV intuition.

All features use only information up to and including `t`, so there is **no leakage**. They are attached to the `X` predictor matrix after timestamp alignment.

## 4. Align Polymarket features to Bloomberg's 5-min grid, assemble `X` and `y`

Polymarket is scraped every ~5 min but the timestamps are **not on the clock** (14:15:52, 14:21:07, …), whereas Bloomberg prices sit exactly on `:00/:05/:10/…`. To merge them I use `merge_asof` with a 5-min tolerance and `direction="backward"`: for every Bloomberg bar, I grab the **latest** Polymarket snapshot strictly at or before that bar. This is the only look-ahead-safe way to align the two clocks.

After alignment we drop rows with no target (the last bar and any pre-warmup AR rows).

### 4.1 Variance pre-filter (conditional) + daily-gap features

Two things happen in this cell:

1. **Variance pre-filter (conditional)** — when `FEATURE_ENGINEERING_DONE = False` in the config cell, we drop near-constants and keep the top-`MAX_FEATURES_PREFILTER` columns by variance (plus all AR features). Once feature engineering / selection has been done upstream, flip the flag to `True` and this filter is skipped. Variance ≠ relevance — this is a stop-gap, not real selection.
2. **Daily-gap features (Approach 2 + 3)** — `is_post_break` is a regime dummy; `poly_movement_during_break` sums `|ΔPolymarket|` across the overnight break and captures the information accumulated while the gold feed was closed. Toggle with `HANDLE_DAILY_GAP`.

## 5. Build `X_traditional`

## 5.1 Dataset construction and registry

`DATASET_SPECS` is the single place that registers every `(X, y)` pair the training loop should iterate over. The combined and ablation datasets are built here so Cell 8 can dispatch the entire grid from one registry.

| Key | X | y | `poly_cols` | `passthrough_cols` |
|-----|---|---|-------------|--------------------|
| `poly` | Polymarket + AR/gap | gold log-return | polymarket cols in `X` | `ar_cols` |
| `trad` | Bloomberg traditional + AR/gap | gold log-return | `None` | unused |
| `poly+trad` | Polymarket + Bloomberg traditional + AR/gap | gold log-return | polymarket cols in `X_combined` | `trad_renamed.columns ∪ combined_ar_cols` |
| `poly_only` | Polymarket only | gold log-return | all of `X_poly_only.columns` | `[]` |
| `bloomberg_only` | Bloomberg traditional only | gold log-return | `None` | unused |
| `ar_only` | Gold AR/gap only | gold log-return | `None` | unused |

Datasets whose `RUN_*` toggle is `False` are not registered.

In [6]:
# %% ── CELL 3-5.1 : BUILD BLOOMBERG TARGET, X, X_traditional, DATASET_SPECS ──
def load_bloomberg(xlsx_path: Path) -> dict[str, pd.DataFrame]:
    """Load every Bloomberg sheet that has a `Date` column.
       Some sheets use `Close`, some use `Last Price` — normalise both to `close`."""
    xl = pd.ExcelFile(xlsx_path)
    out = {}
    for sheet in xl.sheet_names:
        df = pd.read_excel(xlsx_path, sheet_name=sheet, header=0)
        if df.empty or "Date" not in df.columns:
            continue
        df = df.rename(columns={c: c.strip() for c in df.columns})
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df = df.dropna(subset=["Date"]).set_index("Date").sort_index()
        for col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        if "Close" in df.columns:
            df = df.rename(columns={"Close": "close"})
        elif "Last Price" in df.columns:
            df = df.rename(columns={"Last Price": "close"})
        else:
            numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
            if not numeric_cols:
                continue
            df = df.rename(columns={numeric_cols[-1]: "close"})
        out[sheet] = df
    return out


bb = load_bloomberg(BLOOMBERG_XLSX)
print("Bloomberg sheets loaded:", list(bb.keys()))
gold = bb[TARGET_SHEET][["close"]].rename(columns={"close": "gold_close"})
print("Gold close range:", gold.index.min(), "→", gold.index.max(), "| rows:", len(gold))

# %% ── CELL 3 : BUILD y ──────────────────────────────────────────────────────
gold["logret_bar"] = np.log(gold["gold_close"]).diff()
gold["logret_horizon"] = (
    np.log(gold["gold_close"]) - np.log(gold["gold_close"]).shift(HORIZON_STEPS)
)
gold["y"] = gold["logret_horizon"].shift(-HORIZON_STEPS)
gold["logret_5m"] = gold["logret_bar"]
print(f"Target horizon: {RETURN_HORIZON_MIN} min  ({HORIZON_STEPS} bars of {BAR_MINUTES} min each)")
print(gold[["gold_close", "logret_bar", "logret_horizon", "y"]].tail(6))

# %% ── CELL 3.1 : AR FEATURES ────────────────────────────────────────────────
def build_ar_features(logret: pd.Series,
                      lags=AR_LAGS,
                      ma_windows=AR_MA_WINDOWS,
                      vol_windows=(12, 36)) -> pd.DataFrame:
    feats = {}
    for lag in lags:
        feats[f"ar_ret_lag{lag}"] = logret.shift(lag)
    for window in ma_windows:
        feats[f"ar_ret_ma{window}"] = logret.shift(1).rolling(window).mean()
    for window in vol_windows:
        feats[f"ar_ret_std{window}"] = logret.shift(1).rolling(window).std()
    return pd.DataFrame(feats)


ar_feats = build_ar_features(gold["logret_bar"])
print("AR feature matrix shape:", ar_feats.shape)
print(ar_feats.tail(3))

# %% ── CELL 4 : ALIGN + BUILD X, y ───────────────────────────────────────────
start, end = X_raw.index.min().floor("5min"), X_raw.index.max().ceil("5min")
gold_aligned = gold.loc[(gold.index >= start) & (gold.index <= end)].copy()
print(f"Bloomberg bars in overlap window: {len(gold_aligned)}")

X_raw_sorted = X_raw.sort_index()
merged = pd.merge_asof(
    left=gold_aligned.reset_index().rename(columns={"Date": "ts"}),
    right=X_raw_sorted.reset_index().rename(columns={"scraped_at": "ts"}),
    on="ts",
    direction="backward",
    tolerance=pd.Timedelta(f"{BAR_MINUTES}min"),
).set_index("ts")

merged = merged.join(ar_feats, how="left")
y = merged["y"]
polymarket_cols = list(X_raw.columns)
ar_cols = list(ar_feats.columns)
feature_cols = polymarket_cols + ar_cols
X = merged[feature_cols].copy()

valid = y.notna() & merged[ar_cols].notna().all(axis=1)
X, y = X.loc[valid], y.loc[valid]
X[polymarket_cols] = X[polymarket_cols].ffill().fillna(0.0)

print(f"Final X shape: {X.shape}")
print(f"Final y shape: {y.shape}")
print(
    f"y sample stats: mean={y.mean():.2e}  std={y.std():.2e}  "
    f"min={y.min():.2e}  max={y.max():.2e}"
)

# %% ── CELL 4.1 : VARIANCE PREFILTER + DAILY-GAP FEATURES ───────────────────
if FEATURE_ENGINEERING_DONE:
    print("🆗 FEATURE_ENGINEERING_DONE=True -> skipping variance prefilter.")
    X_all = X.copy()
    print(f"X shape (no prefilter): {X.shape}")
else:
    vt = VarianceThreshold(threshold=1e-8)
    vt.fit(X.values)
    kept_mask = vt.get_support()
    kept_cols = X.columns[kept_mask]
    print(f"After VarianceThreshold: {len(kept_cols)} / {X.shape[1]} columns survive.")

    variances = X[kept_cols].var().sort_values(ascending=False)
    top_cols = variances.head(MAX_FEATURES_PREFILTER).index.tolist()
    for col in ar_cols:
        if col in X.columns and col not in top_cols:
            top_cols.append(col)

    X_all = X.copy()
    X = X[top_cols]
    print(f"X shape after prefilter : {X.shape}")

polymarket_cols_kept = [c for c in X.columns if c not in ar_cols]
print(f"  polymarket cols kept: {len(polymarket_cols_kept)}  |  AR cols: {len(ar_cols)}")

if HANDLE_DAILY_GAP:
    _gap_step = X.index.to_series().diff()
    _is_post_break = (_gap_step > pd.Timedelta(GAP_THRESHOLD)).astype(int)
    X["is_post_break"] = _is_post_break.values

    _poly_abs_delta = X_raw.sort_index().diff().abs().sum(axis=1)
    _gap_feature = pd.Series(0.0, index=X.index)
    _times = X.index.to_list()
    for i in range(1, len(_times)):
        t_prev, t_curr = _times[i - 1], _times[i]
        if (t_curr - t_prev) > pd.Timedelta(GAP_THRESHOLD):
            _mask = (_poly_abs_delta.index > t_prev) & (_poly_abs_delta.index <= t_curr)
            _gap_feature.iloc[i] = float(_poly_abs_delta.loc[_mask].sum())
    X["poly_movement_during_break"] = _gap_feature.values

    ar_cols = list(ar_cols) + ["is_post_break", "poly_movement_during_break"]
    print(
        f"✅ Daily-gap handling ON.  post-break bars: {int(_is_post_break.sum())}  |  "
        f"non-zero 'poly_movement_during_break': {int((_gap_feature != 0).sum())}"
    )
    print(f"  new X shape: {X.shape}")
else:
    print("ℹ️  Daily-gap handling OFF (HANDLE_DAILY_GAP=False).")

poly_feature_cols = [c for c in X.columns if c not in ar_cols]
print(f"Polymarket feature cols : {len(poly_feature_cols)}")
print(f"AR / gap cols           : {len(ar_cols)}")

# %% ── CELL 5 : X_traditional ────────────────────────────────────────────────
X_traditional = build_X_traditional(
    bb_dict=bb,
    target_sheet=TARGET_SHEET,
    master_index=gold.index,
    align_index=X.index,
    max_ffill_bars=TRADITIONAL_MAX_FFILL_BARS,
    use_staleness=TRADITIONAL_USE_STALENESS,
    ar_lags=TRADITIONAL_AR_LAGS,
    ar_ma_windows=AR_MA_WINDOWS,
    ar_vol_windows=[12, 36],
    logret_bar=gold["logret_bar"],
    stationarity_mode=TRADITIONAL_STATIONARITY_MODE,
    stationary_panel_path=BB_STATIONARY_PANEL_PATH,
    stationary_metadata=BB_STATIONARITY_METADATA,
)
y_traditional = y.loc[X_traditional.index]

if HANDLE_DAILY_GAP:
    _trad_gap_step = X_traditional.index.to_series().diff()
    _trad_is_post_break = (_trad_gap_step > pd.Timedelta(GAP_THRESHOLD)).astype(int)
    X_traditional["is_post_break"] = _trad_is_post_break.values

    _trad_gap_feature = pd.Series(0.0, index=X_traditional.index)
    _trad_times = X_traditional.index.to_list()
    for i in range(1, len(_trad_times)):
        t_prev, t_curr = _trad_times[i - 1], _trad_times[i]
        if (t_curr - t_prev) > pd.Timedelta(GAP_THRESHOLD):
            _mask = (gold["logret_bar"].index > t_prev) & (gold["logret_bar"].index <= t_curr)
            _trad_gap_feature.iloc[i] = float(gold["logret_bar"].loc[_mask].abs().sum())
    X_traditional["trad_movement_during_break"] = _trad_gap_feature.values

    print(
        f"✅ Trad daily-gap features added.  post-break bars: {int(_trad_is_post_break.sum())}  |  "
        f"non-zero 'trad_movement_during_break': {int((_trad_gap_feature != 0).sum())}"
    )

trad_ar_cols = [
    c for c in X_traditional.columns
    if c.startswith("trad_gold_lag")
    or c.startswith("trad_gold_ma")
    or c.startswith("trad_gold_std")
    or c in ("is_post_break", "trad_movement_during_break")
]
trad_feature_cols = [c for c in X_traditional.columns if c not in trad_ar_cols]

print(f"X_traditional shape       : {X_traditional.shape}")
print(f"y_traditional shape       : {y_traditional.shape}")
print(f"  traditional feature cols: {len(trad_feature_cols)}")
print(f"  AR / gap cols           : {len(trad_ar_cols)}")
print(f"  Features preview        : {list(X_traditional.columns[:8])}")

# %% ── CELL 5.1 : DATASET CONSTRUCTION AND REGISTRY ─────────────────────────
assert X.index.equals(X_traditional.index), (
    "X and X_traditional indices don't match — re-run Cells 4-5 before Cell 5.1."
)

poly_cols = [c for c in X.columns if c not in ar_cols]
poly_passthrough_cols = [c for c in ar_cols if c in X.columns]

trad_renamed = X_traditional.add_prefix("trad__")
trad_ar_cols_prefixed = [f"trad__{c}" for c in trad_ar_cols]

X_combined = pd.concat([X, trad_renamed], axis=1)
y_combined = y.copy()
X_combined = X_combined.dropna(axis=1, how="all")
X_combined = X_combined.ffill().fillna(0.0)

combined_ar_cols = [c for c in poly_passthrough_cols if c in X_combined.columns]
combined_ar_cols += [c for c in trad_ar_cols_prefixed if c in X_combined.columns]
combined_ar_cols = list(dict.fromkeys(combined_ar_cols))

poly_combined_cols = [c for c in poly_cols if c in X_combined.columns]
combined_passthrough_cols = list(dict.fromkeys(
    [c for c in trad_renamed.columns if c in X_combined.columns] + combined_ar_cols
))

poly_only_cols = [
    c for c in X.columns
    if c not in ar_cols
    and c not in ("is_post_break", "poly_movement_during_break")
]
X_poly_only = X[poly_only_cols].copy()
y_poly_only = y.copy()

bloomberg_only_cols = [c for c in X_traditional.columns if c not in trad_ar_cols]
X_bloomberg_only = X_traditional[bloomberg_only_cols].copy()
y_bloomberg_only = y_traditional.copy()

ar_only_cols = [c for c in ar_cols if c in X.columns]
X_ar_only = X[ar_only_cols].copy()
y_ar_only = y.copy()


def register_dataset(registry, enabled, key, X_data, y_data, poly_cols_value=None, passthrough_cols_value=None):
    if not enabled:
        return
    registry[key] = {
        "X": X_data,
        "y": y_data,
        "poly_cols": list(poly_cols_value) if poly_cols_value is not None else None,
        "passthrough_cols": list(passthrough_cols_value) if passthrough_cols_value is not None else None,
    }


DATASET_SPECS = {}
register_dataset(DATASET_SPECS, RUN_POLY, "poly", X, y, poly_cols, poly_passthrough_cols)
register_dataset(DATASET_SPECS, RUN_TRAD, "trad", X_traditional, y_traditional, None, None)
register_dataset(
    DATASET_SPECS,
    RUN_COMBINED,
    "poly+trad",
    X_combined,
    y_combined,
    poly_combined_cols,
    combined_passthrough_cols,
)
register_dataset(
    DATASET_SPECS,
    RUN_POLY_ONLY,
    "poly_only",
    X_poly_only,
    y_poly_only,
    list(X_poly_only.columns),
    [],
)
register_dataset(DATASET_SPECS, RUN_BLOOMBERG_ONLY, "bloomberg_only", X_bloomberg_only, y_bloomberg_only, None, None)
register_dataset(DATASET_SPECS, RUN_AR_ONLY, "ar_only", X_ar_only, y_ar_only, None, None)

print("\nRegistered DATASET_SPECS:")
for tag, ds in DATASET_SPECS.items():
    poly_count = 0 if ds["poly_cols"] is None else len(ds["poly_cols"])
    passthrough_count = 0 if ds["passthrough_cols"] is None else len(ds["passthrough_cols"])
    print(
        f"  [{tag:15s}]  X={ds['X'].shape}  y={ds['y'].shape}  "
        f"poly_cols={poly_count}  passthrough={passthrough_count}"
    )

Bloomberg sheets loaded: ['GOLD USD SPOT PER OZ', 'CRUDE OIL (WTI) FUTURES PRICE', 'CRUDE OIL (BRENT) FUTURES PRICE', 'S&P 500 Index', 'USD Index', 'VIX Index', 'USGG10YR Index', 'EURUSD', 'GC1 Comdty', 'GLD US Equity']
Gold close range: 2026-04-01 00:00:00 → 2026-05-08 22:55:00 | rows: 7459
Target horizon: 60 min  (12 bars of 5 min each)
                     gold_close  logret_bar  logret_horizon   y
Date                                                           
2026-05-08 22:30:00     4717.14   -0.000049       -0.001286 NaN
2026-05-08 22:35:00     4717.88    0.000157       -0.001207 NaN
2026-05-08 22:40:00     4716.74   -0.000242       -0.001834 NaN
2026-05-08 22:45:00     4715.54   -0.000254       -0.001883 NaN
2026-05-08 22:50:00     4715.14   -0.000085       -0.002036 NaN
2026-05-08 22:55:00     4715.25    0.000023       -0.001911 NaN
AR feature matrix shape: (7459, 11)
                     ar_ret_lag1  ar_ret_lag2  ar_ret_lag3  ar_ret_lag6  \
Date                                

## 6. Modelling framework (walk-forward)

Everything below is organised so that shared factories live in one place and each dataset builds a small `MODEL_REGISTRY` from them.

```python
BASE_MODEL_BUILDERS = {
    "linear":   lambda ds: make_linear(),
    "rf":       lambda ds: make_rf(),
    "lstm":     lambda ds: make_lstm(),
    "lasso_cv": lambda ds: make_lasso_cv(),
}
MODEL_REGISTRY = build_model_registry(dataset_tag, dataset_spec)
```

A *model factory* returns an object exposing `.fit(X,y)` and `.predict(X)`. The walk-forward loop is identical for all of them; all LSTM-family models (`lstm` and `lstm_pls`) use the **batched retraining** path (one refit every `LSTM_TEST_BLOCK` obs), while all non-LSTM models use `step=1`.

### Cross-validation strategy
For each `(model, window_scheme)` combo we run an **expanding or fixed-size walk-forward** where at each step the model is trained on `[t-window, t-1]` (fixed) or `[start, t-1]` (expanding) and predicts `y_t`. This is the closest scikit-learn analogue to your "train on rolling window, test on training_size+1" spec. For LSTM-family models (`lstm` and `lstm_pls`) we predict 12 steps in one go before refitting — same walk-forward, just coarser grid.

In [7]:
# %% ── CELL 6.1 : MODEL FACTORIES ────────────────────────────────────────────
# Shared model factories live here; dataset-specific routing happens in
# build_model_registry() via DATASET_SPECS['poly_cols'] and ['passthrough_cols'].

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as _SKPipeline
from sklearn.linear_model import LinearRegression as _LinearRegression, LassoCV as _LassoCV
from sklearn.ensemble import RandomForestRegressor as _RFR
from sklearn.cross_decomposition import PLSRegression as _PLS
from sklearn.model_selection import TimeSeriesSplit as _TSS
from sklearn.feature_selection import VarianceThreshold as _VT
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

MAX_SAFE_PLS_COMPONENTS = 15
MODELS_REQUIRING_POLY_BLOCK = {"pls_ols", "rf_pls", "lstm_pls"}


class PLSTransformer(BaseEstimator, TransformerMixin):
    """PLS used purely as a supervised dimensionality-reduction transformer.

    Composes sklearn's PLSRegression but exposes ONLY the X-side scores, so it
    is safe as a step inside a Pipeline / ColumnTransformer.

    Why this exists: PLSRegression.fit_transform(X, y) internally calls
    transform(X, y), and PLSRegression.transform(X, y) returns a
    (x_scores, y_scores) TUPLE. When raw PLSRegression is the final step of a
    Pipeline nested in a ColumnTransformer, that branch emits the tuple at fit
    time and ColumnTransformer._validate_output rejects it ("output ... should
    be 2D"). This class deliberately does NOT override fit_transform, so it
    inherits TransformerMixin.fit_transform, which calls transform(X) with no
    y -> always a 2-D array of X-scores.
    """

    def __init__(self, n_components=2, max_iter=500, scale=False):
        # Store every constructor argument verbatim and UNMODIFIED. This is
        # required so sklearn.clone() round-trips the estimator correctly
        # (walk-forward refits/clones the model each window). Do NOT coerce,
        # clamp, cast, or otherwise transform these values here.
        self.n_components = n_components
        self.max_iter = max_iter
        self.scale = scale

    def fit(self, X, y=None):
        self.pls_ = _PLS(
            n_components=self.n_components,
            max_iter=self.max_iter,
            scale=self.scale,
        )
        self.pls_.fit(X, y)
        return self

    def transform(self, X):
        check_is_fitted(self, "pls_")
        # transform(X) with no y -> X-scores only -> 2-D array.
        return self.pls_.transform(X)

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, "pls_")
        return np.asarray(
            [f"pls{i}" for i in range(self.pls_.n_components)],
            dtype=object,
        )


def make_linear():
    """Scaled OLS baseline on the raw feature block."""
    return _SKPipeline([
        ("scaler", StandardScaler()),
        ("ols", _LinearRegression()),
    ])


def make_rf():
    """Random Forest on the raw feature block."""
    return _RFR(
        n_estimators=RF_N_ESTIMATORS,
        max_depth=None,
        min_samples_leaf=5,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )


def _build_lstm(n_features: int, seq_len: int = LSTM_SEQ_LEN):
    model = Sequential([
        Input(shape=(seq_len, n_features)),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(1, activation="linear"),
    ])
    model.compile(optimizer="adam", loss="mse")
    return model


class LSTMRegressor:
    """LSTM with internal X/y scaling and a sliding-window transformer."""
    def __init__(self, seq_len=LSTM_SEQ_LEN, epochs=LSTM_EPOCHS, batch=LSTM_BATCH):
        self.seq_len, self.epochs, self.batch = seq_len, epochs, batch
        self.xs, self.ys = StandardScaler(), StandardScaler()
        self.model = None
        self._last_train_X = None

    def _seq(self, Xs, ys=None):
        X_out, y_out = [], []
        for i in range(self.seq_len, len(Xs)):
            X_out.append(Xs[i-self.seq_len:i])
            if ys is not None:
                y_out.append(ys[i])
        X_out = np.asarray(X_out)
        return (X_out, np.asarray(y_out)) if ys is not None else X_out

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y).reshape(-1, 1)
        Xs = self.xs.fit_transform(X)
        ys = self.ys.fit_transform(y).ravel()
        Xseq, yseq = self._seq(Xs, ys)
        self.model = _build_lstm(X.shape[1], self.seq_len)
        es = EarlyStopping(patience=3, restore_best_weights=True, monitor="loss")
        self.model.fit(
            Xseq,
            yseq,
            epochs=self.epochs,
            batch_size=self.batch,
            verbose=0,
            callbacks=[es],
        )
        self._last_train_X = Xs[-self.seq_len:]
        return self

    def predict(self, X):
        X = np.asarray(X)
        Xs = self.xs.transform(X)
        Xs_ext = np.vstack([self._last_train_X, Xs]) if self._last_train_X is not None else Xs
        starts = range(self.seq_len, len(Xs_ext))
        if len(starts) == 0:
            return np.asarray([], dtype=float)
        windows = np.stack([Xs_ext[i-self.seq_len:i] for i in starts]).astype("float32")
        preds = np.asarray(self.model(windows, training=False)).reshape(-1)
        return self.ys.inverse_transform(preds.reshape(-1, 1)).ravel()


def make_lstm():
    if not _TF_AVAILABLE:
        return None
    return LSTMRegressor()


def make_pls_preprocessor(n_components: int, poly_cols: list, passthrough_cols: list):
    """Apply PLS only to the polymarket block and pass everything else through."""
    poly_cols = list(poly_cols or [])
    passthrough_cols = list(passthrough_cols or [])
    if not poly_cols:
        raise ValueError("PLS preprocessing requires at least one polymarket column.")

    safe_n = min(int(n_components), MAX_SAFE_PLS_COMPONENTS)
    transformers = [
        (
            "poly_pls",
            _SKPipeline([
                ("vt", _VT(threshold=1e-8)),
                ("scaler", StandardScaler()),
                ("pls", PLSTransformer(n_components=safe_n, max_iter=500, scale=False)),
            ]),
            poly_cols,
        ),
    ]
    if passthrough_cols:
        transformers.append(("passthru", "passthrough", passthrough_cols))
    return ColumnTransformer(transformers)


def make_pls_ols(n_components: int, poly_cols: list, passthrough_cols: list):
    return _SKPipeline([
        ("preproc", make_pls_preprocessor(n_components, poly_cols, passthrough_cols)),
        ("ols", _LinearRegression()),
    ])


def make_lasso_cv():
    return _SKPipeline([
        ("scaler", StandardScaler()),
        ("lasso", _LassoCV(
            cv=_TSS(n_splits=3),
            n_alphas=20,
            max_iter=10_000,
            random_state=RANDOM_STATE,
        )),
    ])


def make_rf_pls(n_components: int, poly_cols: list, passthrough_cols: list):
    return _SKPipeline([
        ("preproc", make_pls_preprocessor(n_components, poly_cols, passthrough_cols)),
        ("rf", _RFR(
            n_estimators=RF_N_ESTIMATORS,
            max_depth=None,
            min_samples_leaf=5,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ])


class LSTMPLSWrapper:
    """PLS-preprocessed LSTM wrapper for Keras sequence models."""
    def __init__(self, n_components: int, poly_cols: list, passthrough_cols: list,
                 seq_len=LSTM_SEQ_LEN, epochs=LSTM_EPOCHS, batch=LSTM_BATCH):
        self.n_components = n_components
        self.poly_cols = list(poly_cols or [])
        self.passthrough_cols = list(passthrough_cols or [])
        self.seq_len = seq_len
        self.epochs = epochs
        self.batch = batch
        self.transformer = None
        self.core = None

    def fit(self, X, y):
        self.transformer = make_pls_preprocessor(
            self.n_components,
            self.poly_cols,
            self.passthrough_cols,
        )
        Z = self.transformer.fit_transform(X, y)
        self.core = LSTMRegressor(
            seq_len=self.seq_len,
            epochs=self.epochs,
            batch=self.batch,
        )
        self.core.fit(Z, y)
        return self

    def predict(self, X):
        Z = self.transformer.transform(X)
        return self.core.predict(Z)


BASE_MODEL_BUILDERS = {}
if RUN_LINEAR:
    BASE_MODEL_BUILDERS["linear"] = lambda ds: make_linear()
if RUN_RF:
    BASE_MODEL_BUILDERS["rf"] = lambda ds: make_rf()
if RUN_LSTM and _TF_AVAILABLE:
    BASE_MODEL_BUILDERS["lstm"] = lambda ds: make_lstm()
if RUN_LASSO_CV:
    BASE_MODEL_BUILDERS["lasso_cv"] = lambda ds: make_lasso_cv()
if RUN_PLS_OLS:
    BASE_MODEL_BUILDERS["pls_ols"] = (
        lambda ds: make_pls_ols(PLS_N_COMPONENTS, ds["poly_cols"], ds["passthrough_cols"])
    )
if RUN_RF_PLS:
    BASE_MODEL_BUILDERS["rf_pls"] = (
        lambda ds: make_rf_pls(PLS_N_COMPONENTS, ds["poly_cols"], ds["passthrough_cols"])
    )
if RUN_LSTM_PLS and _TF_AVAILABLE:
    BASE_MODEL_BUILDERS["lstm_pls"] = (
        lambda ds: LSTMPLSWrapper(PLS_N_COMPONENTS, ds["poly_cols"], ds["passthrough_cols"])
    )

if RUN_LSTM and not _TF_AVAILABLE:
    print("⚠️  RUN_LSTM=True but TensorFlow is unavailable — skipping `lstm` registration.")
if RUN_LSTM_PLS and not _TF_AVAILABLE:
    print("⚠️  RUN_LSTM_PLS=True but TensorFlow is unavailable — skipping `lstm_pls` registration.")


def build_model_registry(dataset_tag: str, dataset_spec: dict) -> dict:
    registry = {}
    poly_cols = dataset_spec.get("poly_cols")
    has_poly_block = poly_cols is not None and len(poly_cols) > 0

    for model_name, builder in BASE_MODEL_BUILDERS.items():
        if model_name in MODELS_REQUIRING_POLY_BLOCK:
            if FE_INPUT_MODE != "raw_panel":
                print(
                    f"  ⏭ Not registering {model_name} for dataset '{dataset_tag}' "
                    f"(PLS variants are disabled in FE_INPUT_MODE={FE_INPUT_MODE!r})."
                )
                continue
            if not has_poly_block:
                print(
                    f"  ⏭ Not registering {model_name} for dataset '{dataset_tag}' "
                    "(requires a polymarket block)."
                )
                continue
        registry[model_name] = lambda builder=builder, ds=dataset_spec: builder(ds)

    return registry


print(f"Base model builders: {list(BASE_MODEL_BUILDERS.keys())}")
print("✅ Shared model factories loaded.")

Base model builders: ['lstm_pls']
✅ Shared model factories loaded.


### 6.2 Walk-forward runner

All non-LSTM models use `step=1` (one-step-ahead, the finest grid possible). All LSTM-family models (`lstm` and `lstm_pls`) use `step=LSTM_TEST_BLOCK` (=12, ~1 hour per refit). The same helper covers both.

In [8]:
# %% ── CELL 6.2 : WALK-FORWARD RUNNER ────────────────────────────────────────
def walk_forward_indices(n: int, kind: str, size: int, step: int = 1):
    assert kind in ("fixed", "expanding")
    start = size
    for i in range(start, n, step):
        tr = slice(i - size, i) if kind == "fixed" else slice(0, i)
        te = slice(i, min(i + step, n))
        if te.stop <= te.start:
            break
        yield tr, te


def compute_metrics(y_true, y_pred) -> dict:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return {
        "n": int(len(y_true)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)) if len(y_true) > 1 else float("nan"),
        "dir_acc": float(np.mean(np.sign(y_true) == np.sign(y_pred))),
    }


def walk_forward(model_name: str, scheme_name: str, X: pd.DataFrame, y: pd.Series, model_factory):
    kind, size = WINDOW_SCHEMES[scheme_name]
    step = LSTM_TEST_BLOCK if model_name.startswith("lstm") else 1
    y_true, y_pred, ts_pred = [], [], []
    train_time_s = 0.0
    iters = list(walk_forward_indices(len(X), kind, size, step=step))

    for k, (tr, te) in enumerate(iters):
        model = model_factory()
        if model is None:
            return None
        t0 = time.perf_counter()
        model.fit(X.iloc[tr], y.iloc[tr])
        train_time_s += time.perf_counter() - t0
        pred = model.predict(X.iloc[te])
        y_true.extend(y.iloc[te].tolist())
        y_pred.extend(np.asarray(pred).tolist())
        ts_pred.extend(y.iloc[te].index.tolist())
        if k % 50 == 0:
            print(
                f"  [{model_name}/{scheme_name}] step {k+1}/{len(iters)} "
                f"(train={tr.stop-tr.start}, test={te.stop-te.start})"
            )

    return {
        "y_true": np.asarray(y_true),
        "y_pred": np.asarray(y_pred),
        "timestamps": ts_pred,
        "final_model": model,
        "train_time_s": train_time_s,
    }


## 7. Logging and model persistence

- `Results/runs_log.txt` — one human-readable row per `(dataset, model, window)` run.
- `Results/runs_log.jsonl` — the same info as JSON for reproducibility / downstream comparison across weeks.
- `Models/<MODEL>_<WINDOW>_<DATASET>_h<HORIZON>m_f<NFEAT>_<DATA-DATE>.(pkl|keras)` — the **final-step** fitted model (trained on the last available window), so you can reload it without rerunning the whole walk-forward.

**"Retrain only if data changed"** logic:
- We compute `sha1` of the input gold panel CSV (content, not filename) and store it inside the sidecar JSON. On the next run, if a cached model for that run configuration exists and `FORCE_RETRAIN is False`, we just reuse it.

In [9]:
# %% ── CELL 7 : PERSISTENCE HELPERS ─────────────────────────────────────────
def file_sha1(path: Path, bufsize=1<<20) -> str:
    h = hashlib.sha1()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(bufsize), b""):
            h.update(chunk)
    return h.hexdigest()


DATA_HASH = file_sha1(panel_path)
print("Data SHA1:", DATA_HASH[:12], "…")


def artefact_paths(model_name: str, scheme: str, data_date: str, n_features: int,
                   dataset_tag: str = "poly") -> tuple[Path, Path]:
    tokens = [
        model_name.upper(),
        scheme,
        dataset_tag,
        f"h{RETURN_HORIZON_MIN}m",
        f"f{n_features}",
        data_date,
    ]
    stem = "_".join(tokens)
    ext = ".keras" if model_name.startswith("lstm") else ".pkl"
    return MODELS_DIR / f"{stem}{ext}", RESULTS_DIR / f"{stem}.json"


def cached_run_valid(meta_path: Path, data_hash: str) -> bool:
    if not meta_path.exists():
        return False
    pred_path = meta_path.with_name(meta_path.stem + ".predictions.csv")
    if not pred_path.exists():
        return False  # force recompute so the prediction sidecar is generated
    try:
        meta = json.loads(meta_path.read_text())
        return meta.get("data_sha1") == data_hash and not FORCE_RETRAIN
    except Exception:
        return False


def save_artefacts(model_name, scheme, data_date, data_hash, metrics, result,
                   X_cols, train_time_s=None, dataset_tag="poly"):
    """Persist model, metadata, and walk-forward predictions, then append to the master run logs.

    Metadata schema changed in this refactor: legacy linear-model and
    dimensionality-reduction fields were removed from the run sidecar.
    """
    n_features = len(X_cols)
    model_path, meta_path = artefact_paths(
        model_name,
        scheme,
        data_date,
        n_features,
        dataset_tag=dataset_tag,
    )
    model = result["final_model"]

    if model_name.startswith("lstm"):
        if model_name == "lstm":
            core = model
            transformer = None
        else:
            core = model.core
            transformer = model.transformer
        core.model.save(model_path)
        state = {
            "xs": core.xs,
            "ys": core.ys,
            "seq_len": core.seq_len,
            "last_train_X": core._last_train_X,
        }
        if transformer is not None:
            state["transformer"] = transformer
        joblib.dump(state, model_path.with_suffix(".scalers.pkl"))
    else:
        joblib.dump(model, model_path)

    uses_pls = "pls" in model_name
    trad_dataset = dataset_tag in {"trad", "poly+trad", "bloomberg_only"}
    meta = {
        "model": model_name,
        "window": scheme,
        "dataset_tag": dataset_tag,
        "data_date": data_date,
        "data_sha1": data_hash,
        "run_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
        "train_time_s": round(train_time_s, 3) if train_time_s is not None else None,
        "n_features": n_features,
        "feature_sample": X_cols[:15],
        "metrics": metrics,
        "config": {
            "bar_minutes": BAR_MINUTES,
            "return_horizon_min": RETURN_HORIZON_MIN,
            "horizon_steps": HORIZON_STEPS,
            "ar_lags": AR_LAGS,
            "ar_ma_windows": AR_MA_WINDOWS,
            "feature_engineering_done": FEATURE_ENGINEERING_DONE,
            "prefilter_topN": MAX_FEATURES_PREFILTER if not FEATURE_ENGINEERING_DONE else None,
            "lstm_seq_len": LSTM_SEQ_LEN,
            "lstm_block": LSTM_TEST_BLOCK,
            "lstm_epochs": LSTM_EPOCHS,
            "lstm_batch": LSTM_BATCH,
            "rf_estimators": RF_N_ESTIMATORS,
            "random_state": RANDOM_STATE,
            "handle_daily_gap": HANDLE_DAILY_GAP,
            "gap_threshold": GAP_THRESHOLD,
            "fe_input_mode": FE_INPUT_MODE,
            "trad_max_ffill_bars": TRADITIONAL_MAX_FFILL_BARS if trad_dataset else None,
            "trad_use_staleness": TRADITIONAL_USE_STALENESS if trad_dataset else None,
            "trad_ar_lags": TRADITIONAL_AR_LAGS if trad_dataset else None,
            "pls_n_components": PLS_N_COMPONENTS if uses_pls else None,
            "pls_selection_path": str(PLS_SELECTION_ARTEFACT_PATH) if uses_pls else None,
        },
    }
    meta_path.write_text(json.dumps(meta, indent=2, default=str))

    # ── Persist the walk-forward OOS prediction series for downstream backtesting ──
    pred_path = meta_path.with_name(meta_path.stem + ".predictions.csv")
    assert (
        len(result["timestamps"]) == len(result["y_true"]) == len(result["y_pred"])
    ), "prediction series length mismatch in save_artefacts"
    pd.DataFrame({
        "timestamp": result["timestamps"],
        "y_true":    np.asarray(result["y_true"]),
        "y_pred":    np.asarray(result["y_pred"]),
    }).to_csv(pred_path, index=False)

    train_time_str = f"{train_time_s:.1f}s" if train_time_s is not None else "N/A"
    with open(RESULTS_DIR / "runs_log.txt", "a", encoding="utf-8") as f:
        f.write(
            f"{meta['run_utc']}  {model_name:9s}  {scheme:10s}  {dataset_tag:14s}  "
            f"h={RETURN_HORIZON_MIN}m  f={n_features}  data={data_date}  "
            f"rmse={metrics['rmse']:.5e}  mae={metrics['mae']:.5e}  "
            f"r2={metrics['r2']:+.4f}  dir_acc={metrics['dir_acc']:.3f}  "
            f"train_time={train_time_str}\n"
        )
    with open(RESULTS_DIR / "runs_log.jsonl", "a", encoding="utf-8") as f:
        f.write(json.dumps(meta, default=str) + "\n")

    return meta_path, model_path

Data SHA1: 9e1231a8ad5a …


## 8. Run the full grid

`MODELS × WINDOW_SCHEMES`. Cached artefacts are reused if the data hasn't changed.

In [10]:
# %% ── CELL 8 : RUN THE GRID ─────────────────────────────────────────────────
all_results = []

for dataset_tag, ds in DATASET_SPECS.items():
    X_ds = ds["X"]
    y_ds = ds["y"]
    n_features = X_ds.shape[1]
    MODEL_REGISTRY = build_model_registry(dataset_tag, ds)

    print(f"\n{'='*60}")
    print(f"  Dataset: {dataset_tag}  |  X={X_ds.shape}")
    print(f"  Models : {list(MODEL_REGISTRY.keys())}")
    print(f"{'='*60}")

    for model_name, model_factory in MODEL_REGISTRY.items():
        for scheme in WINDOW_SCHEMES:
            model_path, meta_path = artefact_paths(
                model_name,
                scheme,
                panel_date,
                n_features,
                dataset_tag=dataset_tag,
            )
            if cached_run_valid(meta_path, DATA_HASH):
                meta = json.loads(meta_path.read_text())
                train_time_str = (
                    f"{meta['train_time_s']:.1f}s"
                    if meta.get("train_time_s") is not None
                    else "N/A"
                )
                print(
                    f"✓ Cached: {dataset_tag}/{model_name}/{scheme} — "
                    f"rmse={meta['metrics']['rmse']:.3e}  "
                    f"dir_acc={meta['metrics']['dir_acc']:.3f}  "
                    f"train_time={train_time_str}"
                )
                all_results.append(meta)
                continue

            print(
                f"▶ Training {dataset_tag}/{model_name}/{scheme} …  "
                f"(horizon={RETURN_HORIZON_MIN}m  features={n_features})"
            )
            res = walk_forward(model_name, scheme, X_ds, y_ds, model_factory)
            if res is None:
                continue

            metrics = compute_metrics(res["y_true"], res["y_pred"])
            train_time_s = res["train_time_s"]
            save_artefacts(
                model_name,
                scheme,
                panel_date,
                DATA_HASH,
                metrics,
                res,
                list(X_ds.columns),
                train_time_s,
                dataset_tag=dataset_tag,
            )
            all_results.append({
                "model": model_name,
                "window": scheme,
                "dataset_tag": dataset_tag,
                "data_date": panel_date,
                "n_features": n_features,
                "metrics": metrics,
                "train_time_s": train_time_s,
                "config": {
                    "return_horizon_min": RETURN_HORIZON_MIN,
                    "fe_input_mode": FE_INPUT_MODE,
                    "pls_n_components": PLS_N_COMPONENTS if "pls" in model_name else None,
                },
            })
            print(
                f"   ✓ rmse={metrics['rmse']:.3e}  mae={metrics['mae']:.3e}  "
                f"r2={metrics['r2']:+.4f}  dir_acc={metrics['dir_acc']:.3f}  "
                f"train_time={train_time_s:.1f}s"
            )


  Dataset: poly  |  X=(7134, 703)
  Models : ['lstm_pls']
✓ Cached: poly/lstm_pls/fixed240 — rmse=3.439e-03  dir_acc=0.548  train_time=1363.3s
✓ Cached: poly/lstm_pls/expanding — rmse=3.579e-03  dir_acc=0.522  train_time=4176.8s
  ⏭ Not registering lstm_pls for dataset 'trad' (requires a polymarket block).

  Dataset: trad  |  X=(7134, 31)
  Models : []

  Dataset: poly+trad  |  X=(7134, 734)
  Models : ['lstm_pls']
▶ Training poly+trad/lstm_pls/fixed240 …  (horizon=60m  features=734)
  [lstm_pls/fixed240] step 1/575 (train=240, test=12)
  [lstm_pls/fixed240] step 51/575 (train=240, test=12)
  [lstm_pls/fixed240] step 101/575 (train=240, test=12)
  [lstm_pls/fixed240] step 151/575 (train=240, test=12)
  [lstm_pls/fixed240] step 201/575 (train=240, test=12)
  [lstm_pls/fixed240] step 251/575 (train=240, test=12)
  [lstm_pls/fixed240] step 301/575 (train=240, test=12)
  [lstm_pls/fixed240] step 351/575 (train=240, test=12)
  [lstm_pls/fixed240] step 401/575 (train=240, test=12)
  [lstm_

## 9. Summary table

In [11]:
# %% ── CELL 9 : SUMMARY ─────────────────────────────────────────────────────
summary_rows = []
for result in all_results:
    metrics = result.get("metrics", result)
    cfg = result.get("config", {}) if isinstance(result, dict) else {}
    summary_rows.append({
        "dataset": result.get("dataset_tag", "poly"),
        "model": result.get("model"),
        "window": result.get("window"),
        "date": result.get("data_date"),
        "horizon_min": cfg.get("return_horizon_min", RETURN_HORIZON_MIN),
        "n_features": result.get("n_features"),
        "rmse": metrics.get("rmse"),
        "mae": metrics.get("mae"),
        "r2": metrics.get("r2"),
        "dir_acc": metrics.get("dir_acc"),
        "n": metrics.get("n"),
        "train_time_s": result.get("train_time_s"),
    })

summary_columns = [
    "dataset", "model", "window", "date", "horizon_min", "n_features",
    "rmse", "mae", "r2", "dir_acc", "n", "train_time_s",
]
summary = pd.DataFrame(summary_rows, columns=summary_columns)
if not summary.empty:
    summary = summary.sort_values(["dataset", "model", "window"]).reset_index(drop=True)
print(summary.to_string(index=False))

_summary_snap_name = f"summary_h{RETURN_HORIZON_MIN}m_{panel_date}.csv"
summary.to_csv(RESULTS_DIR / _summary_snap_name, index=False)
print(f"Summary saved to Results/{_summary_snap_name}")

  dataset    model    window       date  horizon_min  n_features     rmse      mae        r2  dir_acc    n  train_time_s
     poly lstm_pls expanding 2026-05-07           60         703 0.003579 0.002493 -0.225574 0.522099 7014   4176.824000
     poly lstm_pls  fixed240 2026-05-07           60         703 0.003439 0.002409 -0.134270 0.548303 6894   1363.296000
poly+trad lstm_pls expanding 2026-05-07           60         734 0.003618 0.002571 -0.252731 0.536498 7014   3599.269968
poly+trad lstm_pls  fixed240 2026-05-07           60         734 0.003299 0.002308 -0.043848 0.568030 6894   1275.414262
poly_only lstm_pls expanding 2026-05-07           60         690 0.003446 0.002402 -0.136112 0.530653 7014   5281.088724
poly_only lstm_pls  fixed240 2026-05-07           60         690 0.003372 0.002350 -0.090069 0.554540 6894   1622.478507
Summary saved to Results/summary_h60m_2026-05-07.csv


In [12]:
# %% ── CELL 9.1 : PLS SENSITIVITY SWEEP ──────────────────────────────────────
sensitivity_results = pd.DataFrame(
    columns=["model", "n_components", "scheme", "rmse", "mae", "r2", "dir_acc", "train_time_s"]
)

if not RUN_PLS_SENSITIVITY_SWEEP:
    print("PLS sensitivity sweep skipped (RUN_PLS_SENSITIVITY_SWEEP=False).")
elif FE_INPUT_MODE != "raw_panel":
    print(
        "PLS sensitivity sweep skipped because FE_INPUT_MODE != 'raw_panel' "
        f"(got {FE_INPUT_MODE!r})."
    )
elif "poly" not in DATASET_SPECS:
    print("PLS sensitivity sweep skipped because the 'poly' dataset is not registered.")
else:
    sensitivity_grid = []
    for n in PLS_SENSITIVITY_GRID:
        n = int(n)
        if n < 1 or n == PLS_N_COMPONENTS or n in sensitivity_grid:
            continue
        sensitivity_grid.append(n)

    if not sensitivity_grid:
        print(
            "PLS sensitivity sweep skipped because no alternate component counts "
            f"remain after excluding the primary value ({PLS_N_COMPONENTS})."
        )
    else:
        poly_spec = DATASET_SPECS["poly"]
        X_sens = poly_spec["X"]
        y_sens = poly_spec["y"]
        poly_cols = poly_spec["poly_cols"]
        passthrough_cols = poly_spec["passthrough_cols"] or []

        sensitivity_registry = {}
        for n in sensitivity_grid:
            if RUN_PLS_OLS:
                sensitivity_registry[f"pls_ols_n{n}"] = (
                    lambda n=n, poly_cols=poly_cols, passthrough_cols=passthrough_cols:
                        make_pls_ols(n, poly_cols, passthrough_cols)
                )
            if RUN_RF_PLS:
                sensitivity_registry[f"rf_pls_n{n}"] = (
                    lambda n=n, poly_cols=poly_cols, passthrough_cols=passthrough_cols:
                        make_rf_pls(n, poly_cols, passthrough_cols)
                )
            if RUN_LSTM_PLS and _TF_AVAILABLE:
                sensitivity_registry[f"lstm_pls_n{n}"] = (
                    lambda n=n, poly_cols=poly_cols, passthrough_cols=passthrough_cols:
                        LSTMPLSWrapper(n, poly_cols, passthrough_cols)
                )

        if not sensitivity_registry:
            print("PLS sensitivity sweep skipped because all PLS model toggles are disabled.")
        else:
            sensitivity_rows = []
            for model_name, model_factory in sensitivity_registry.items():
                n_components = int(model_name.rsplit("_n", 1)[1])
                for scheme in WINDOW_SCHEMES:
                    print(f"▶ Sensitivity run {model_name}/{scheme} …")
                    res = walk_forward(model_name, scheme, X_sens, y_sens, model_factory)
                    if res is None:
                        continue
                    metrics = compute_metrics(res["y_true"], res["y_pred"])
                    sensitivity_rows.append({
                        "model": model_name,
                        "n_components": n_components,
                        "scheme": scheme,
                        "rmse": metrics["rmse"],
                        "mae": metrics["mae"],
                        "r2": metrics["r2"],
                        "dir_acc": metrics["dir_acc"],
                        "train_time_s": res["train_time_s"],
                    })
                    print(
                        f"   ✓ rmse={metrics['rmse']:.3e}  mae={metrics['mae']:.3e}  "
                        f"r2={metrics['r2']:+.4f}  dir_acc={metrics['dir_acc']:.3f}  "
                        f"train_time={res['train_time_s']:.1f}s"
                    )

            sensitivity_results = pd.DataFrame(
                sensitivity_rows,
                columns=["model", "n_components", "scheme", "rmse", "mae", "r2", "dir_acc", "train_time_s"],
            )

            sensitivity_date = dt.datetime.now(dt.timezone.utc).date().isoformat()
            PLS_SELECTION_LOG_DIR.mkdir(parents=True, exist_ok=True)
            sensitivity_csv_path = PLS_SELECTION_LOG_DIR / f"pls_sensitivity_{sensitivity_date}.csv"
            sensitivity_results.to_csv(sensitivity_csv_path, index=False)

            print(f"Sensitivity CSV saved      : {sensitivity_csv_path}")
            if sensitivity_results.empty:
                print("Sensitivity sweep ran but produced no result rows.")
            else:
                sensitivity_summary = (
                    sensitivity_results.assign(
                        model_family=sensitivity_results["model"].str.replace(r"_n\d+$", "", regex=True)
                    )
                    .groupby(["model_family", "n_components"], as_index=False)["rmse"]
                    .mean()
                    .pivot(index="model_family", columns="n_components", values="rmse")
                    .sort_index(axis=0)
                    .sort_index(axis=1)
                )
                print("Mean RMSE across schemes by model family and n_components:")
                print(sensitivity_summary.to_string())

PLS sensitivity sweep skipped (RUN_PLS_SENSITIVITY_SWEEP=False).
